**Benchmarking Script**

Warmup steps: 5 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 99.9 ± 0.2 | 295.4 ± 0.1 | 313.4 ± 0.2 |
| medium | 291.8 ± 0.2 | 870.4 ± 0.4 | 935.5 ± 1.6 |
| large | 617.1 ± 3.3 | 1893.9 ± 29.9 | OOM/ERR |

The standard deviations (see above) are small. Decided not to run the XL and 10B models since they likely will not run on laptop.

Now I ran with 0 warm up steps (my laptop crashed with large model, so I omitted this):

Warmup steps: 0 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 108.6 ± 26.2 | 300.6 ± 12.1 | 319.1 ± 11.2 |
| medium | 296.3 ± 10.0 | 877.0 ± 12.6 | 943.6 ± 15.1 |

As can be seen the times are slightly longer, likely due to cold cache.

And finally running with one warmup step:

Warmup steps: 1 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 99.7 ± 0.1 | 294.8 ± 0.1 | 313.1 ± 0.5 |
| medium | 291.5 ± 0.4 | 867.5 ± 0.1 | 933.7 ± 1.0 |

Even just one warmup step seems to make a difference.

**Nsight Systems Profiling**

Note: I am not using Nsys but rather torch.profile for compatibility with my macbook.

(a) Total time spent for forward is:

| Model | Batch Size | Time (ms) |
|---|---|---|
| small | 256 | 53.80 |
| small | 512 | 115.05 |
| medium | 256 | 145.75 |
| medium | 512 | 319.51 |

For batch size 512, the times without profiling for small (99.9ms) and medium (291.8ms) are slightly smaller as expected.

(b) The `aten::bmm` (batched matrix multiply) kernel takes most time on the forwards pass, consuming ~30% of runtime regardless of model (small/medium). This kernel is called 217 times (medium model) and 109 times (small model). While the forwards pass is dominated by `bmm`, the backwards pass also has `aten::sum` as a significant component, which makes sense since the autograd engine performs lots of summations. (*weird behaviour seen with medium model and 1024 context length: `sum` becomes negligible and `bmm` jumps to ~80%*).

(c) `einsum`, `mean`, `permute`, `max`, `mul`

(d) Compared to inference, when performing a full (forwards/backwards/optimizer) pass, the fraction of time spent on `bmm` falls slightly and time spent on `addcdiv` and `addcmul` rises slightly (which are elementwise operations directly related to steps within the AdamW algorithm).

(e) To perform this comparison, markers were added within the `scaled_dot_product_attention` function. The FLOPs was estimated using analytical formulas. Although the FLOPs ratio (mul FLOPs / softmax FLOPs) is ~85, the runtime ratio is just ~1.3x suggesting softmax is memory bound and that it takes a long time despite relatively little computation.